# UROP MATR Anomaly Detection - Local Version

이 노트북은 로컬 폴더에 있는 프로젝트 코드와 MATR 데이터셋을 바로 사용한다.

기본 경로:

```text
PROJECT_DIR = 현재 작업 폴더, 또는 C:/Users/kyucho/UROP 자동 탐색
MATR_DIR    = PROJECT_DIR/MATR
```

다른 위치를 쓰려면 첫 번째 코드 셀을 실행하기 전에 환경변수 `PROJECT_DIR`, `MATR_DIR`를 지정하면 된다.


## 1. 로컬 경로 설정


In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess

def resolve_project_dir():
    candidates = []

    env_project = os.environ.get('PROJECT_DIR')
    if env_project:
        candidates.append(Path(env_project))

    candidates.extend([
        Path(r'C:/Users/kyucho/UROP'),
        Path.home() / 'UROP',
        Path.cwd(),
    ])

    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if (candidate / 'scripts' / 'run_matr_locked_test_evaluation.py').exists():
            return candidate

    checked = '\n'.join(str(p.expanduser()) for p in candidates)
    raise FileNotFoundError(
        'Could not find project directory containing '
        'scripts/run_matr_locked_test_evaluation.py. Checked:\n'
        + checked
    )

PROJECT_DIR = resolve_project_dir()
WORK_DIR = Path.cwd().resolve()
MATR_DIR = Path(os.environ.get('MATR_DIR', PROJECT_DIR / 'MATR')).expanduser().resolve()
for import_root in [PROJECT_DIR, WORK_DIR]:
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

print('PROJECT_DIR:', PROJECT_DIR)
print('WORK_DIR:', WORK_DIR)
print('MATR_DIR:', MATR_DIR)
print('PROJECT_DIR exists:', PROJECT_DIR.exists())
print('MATR_DIR exists:', MATR_DIR.exists())


## 2. 로컬 파일 및 설정 확인


In [ ]:
# Current SOH configuration:
# - battery-level split
# - sliding-window samples
# - lookback=10
# - scoring horizons=10,50,100 (alpha selection remains H50/H100)
# - H10 reuses the locked configuration/checkpoint but did not participate
#   in the original H50/H100 Optuna hyperparameter-selection objective.
# - target_scale=100, fixed_len=100
CONFIG_DIR = PROJECT_DIR / 'outputs' / 'matr_step7_sliding_l10_optuna_cpdsconv_h50_h100_wide'
CONFIG_PATH = CONFIG_DIR / 'best_optuna_config.json'
TUNING_CONTEXT_PATH = CONFIG_DIR / 'optuna_tuning_config.json'
LOCKED_TEST_DIR = PROJECT_DIR / 'outputs' / 'matr_step7_locked_test_cpdsconv_l10_h10_h50_h100_wide_best'
CHECKPOINT_ROOT = LOCKED_TEST_DIR / 'checkpoints'
SPLIT_MANIFEST_ROOT = LOCKED_TEST_DIR
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'matr_locked_test_sliding_l10_h10_h50_h100_wide_best_anomaly_inference'
RUN_SCRIPT = PROJECT_DIR / 'scripts' / 'run_matr_locked_test_evaluation.py'

# MATR data is expected to be an already-extracted local folder.
pkl_files = sorted(MATR_DIR.rglob('*.pkl')) if MATR_DIR.exists() else []

print('PROJECT_DIR:', PROJECT_DIR)
print('MATR_DIR:', MATR_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('RUN_SCRIPT exists:', RUN_SCRIPT.exists(), RUN_SCRIPT)
print('CONFIG_PATH exists:', CONFIG_PATH.exists(), CONFIG_PATH)
print('TUNING_CONTEXT_PATH exists:', TUNING_CONTEXT_PATH.exists(), TUNING_CONTEXT_PATH)
print('CHECKPOINT_ROOT exists:', CHECKPOINT_ROOT.exists(), CHECKPOINT_ROOT)
print('SPLIT_MANIFEST_ROOT exists:', SPLIT_MANIFEST_ROOT.exists(), SPLIT_MANIFEST_ROOT)
print('PKL file count:', len(pkl_files))
for path in pkl_files[:10]:
    print(path)

if not RUN_SCRIPT.exists():
    raise FileNotFoundError(f'Missing run script: {RUN_SCRIPT}')
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        'Missing tuned config file. Put '
        'outputs/matr_step7_sliding_l10_optuna_cpdsconv_h50_h100_wide/best_optuna_config.json '
        'inside the uploaded project folder.'
    )
if not TUNING_CONTEXT_PATH.exists():
    raise FileNotFoundError(
        'Missing Optuna sidecar config. Put '
        'outputs/matr_step7_sliding_l10_optuna_cpdsconv_h50_h100_wide/optuna_tuning_config.json '
        'next to best_optuna_config.json.'
    )
if not CHECKPOINT_ROOT.is_dir():
    raise FileNotFoundError(f'Missing locked checkpoint directory: {CHECKPOINT_ROOT}')
for seed in [42, 43, 44]:
    manifest_path = SPLIT_MANIFEST_ROOT / f'split_manifest_seed{seed}.json'
    if not manifest_path.exists():
        raise FileNotFoundError(f'Missing original split manifest: {manifest_path}')
if not pkl_files:
    raise RuntimeError(
        f'No .pkl files found under MATR_DIR={MATR_DIR}. '
        'Place the already-extracted MATR dataset folder there, or set os.environ["MATR_DIR"] before running setup.'
    )


## 3. Anomaly score 정의와 α 선택 방법

이 노트북의 열화 이상 점수는 다음과 같이 정의한다.

```text
score = α × degradation_residual_z + (1 - α) × degradation_slope_z
```

실제 이상 정답 라벨이 없으므로 α를 AP/ROC-AUC로 최적화하지 않는다. 각 seed의 validation 배터리를 `alpha_selection` 약 1/3과 `conformal_calibration` 약 2/3으로 분리한다. Component scale과 α 및 horizon scale은 alpha-selection 셀에서만 고정하며, H50/H100의 동일 예측 시작점에서 계산한 셀 위험 순위가 가장 안정적인 α를 선택한다. 그 후 H10/H50/H100이 공통으로 보유한 `input_end_cycle`에서 window score 평균을 horizon severity로 계산하고, 세 horizon의 상대 severity 중앙값을 하나의 연속적인 cell nonconformity score로 사용한다. Calibration 셀은 어떠한 parameter 선택에도 사용하지 않고 test cell score의 empirical conformal-style p-value를 계산하는 기준 집단으로만 사용한다. 따라서 window q95, 이상 window 비율, 2-of-3 binary vote는 사용하지 않는다. p-value는 이상일 확률이 아니라 validation reference 집단에서의 상대적 희귀도이며, 기존 predictor가 validation model selection에 사용됐으므로 탐색적 empirical p-value로 해석한다. 셀별 관측 수명 차이가 점수를 지배하는지도 별도의 coverage sensitivity audit로 함께 확인한다.


In [ ]:
import numpy as np
import pandas as pd

from scripts.matr_anomaly_scoring import (
    AnomalyConfig,
    add_scores_by_seed,
    aggregate_cell_nonconformity,
    aggregate_physical_cell_evidence,
    audit_cell_score_coverage,
    apply_component_calibration,
    apply_empirical_conformal_pvalues,
    apply_horizon_severity_calibration,
    assert_seed_split_isolation,
    fit_component_calibration,
    fit_horizon_severity_calibration,
    plot_alpha_search,
    prepare_residual_features,
    select_alpha_by_seed,
    split_validation_roles,
    summarize_common_horizon_scores,
)

ANOMALY_CONFIG = AnomalyConfig(
    target_model='cpmlp_cpdsconv_fusion',
    expected_horizons=(10, 50, 100),
    alpha_selection_horizons=(50, 100),
    alpha_grid=tuple(float(x) for x in np.round(np.linspace(0.0, 1.0, 21), 2)),
    validation_selection_fraction=1.0 / 3.0,
    alpha_cell_aggregation='mean',
    min_common_windows=5,
    min_paired_cells=5,
    bootstrap_repeats=500,  # 최종 보고에서는 1000 이상 권장
    conformal_candidate_p=0.10,
    conformal_strong_p=0.05,
    random_state=20260711,
    threshold_method='cell_empirical_conformal',
)

print('Anomaly score: alpha * residual_z + (1 - alpha) * slope_z')
print('Alpha grid:', ANOMALY_CONFIG.alpha_grid)
print('Scoring horizons:', ANOMALY_CONFIG.expected_horizons)
print('Alpha selection horizons:', ANOMALY_CONFIG.alpha_selection_horizons)
print('Threshold method:', ANOMALY_CONFIG.threshold_method)
print('Cell score: median of H10/H50/H100 relative mean severities')
print('Candidate p <=', ANOMALY_CONFIG.conformal_candidate_p)
print('Strong candidate p <=', ANOMALY_CONFIG.conformal_strong_p)


## 4. GPU 확인


In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('DEVICE:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 5. 최신 코드 실행


In [ ]:
help_result = subprocess.run(
    [sys.executable, str(RUN_SCRIPT), '--help'],
    cwd=PROJECT_DIR,
    text=True,
    capture_output=True,
)

help_text = help_result.stdout + '\n' + help_result.stderr
required_options = [
    '--sample-mode', '--lookback', '--horizons', '--include-references',
    '--inference-only', '--checkpoint-root', '--split-manifest-root',
    '--write-validation-predictions',
]
missing_options = [opt for opt in required_options if opt not in help_text]
if missing_options:
    raise RuntimeError(
        'The evaluation script is not the updated version. Missing options: '
        + ', '.join(missing_options)
        + '\n\nSTDERR:\n'
        + help_result.stderr[-4000:]
    )

config_rel = CONFIG_PATH.relative_to(PROJECT_DIR).as_posix()
output_rel = OUTPUT_DIR.relative_to(PROJECT_DIR).as_posix()
checkpoint_root_rel = CHECKPOINT_ROOT.relative_to(PROJECT_DIR).as_posix()
split_manifest_root_rel = SPLIT_MANIFEST_ROOT.relative_to(PROJECT_DIR).as_posix()

cmd = [
    sys.executable,
    str(RUN_SCRIPT),
    '--data-root', str(MATR_DIR),
    '--config-path', config_rel,
    '--output-dir', output_rel,
    '--checkpoint-root', checkpoint_root_rel,
    '--split-manifest-root', split_manifest_root_rel,
    '--inference-only',
    '--write-validation-predictions',
    '--device', DEVICE,
    '--include-references',
    '--sample-mode', 'sliding-window',
    '--lookback', '10',
    '--horizons', '10', '50', '100',
    '--seeds', '42', '43', '44',
    '--target-scale', '100',
    '--fixed-len', '100',
]

print('Running command:')
print(' '.join(cmd))

result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print('return code:', result.returncode)
print('\n--- STDOUT tail ---')
print(result.stdout[-8000:])
print('\n--- STDERR tail ---')
print(result.stderr[-8000:])

if result.returncode != 0:
    raise RuntimeError('Model evaluation command failed. Check STDERR above.')


## 6. 필수 결과 파일 확인


In [ ]:
import json

required_outputs = [
    OUTPUT_DIR / 'locked_test_summary.csv',
    OUTPUT_DIR / 'test_predictions.csv',
    OUTPUT_DIR / 'validation_predictions.csv',
    OUTPUT_DIR / 'test_summary_by_model_horizon.csv',
    OUTPUT_DIR / 'locked_test_config.json',
]

for path in required_outputs:
    print(path.name, path.exists(), path)

missing = [path for path in required_outputs if not path.exists()]
if missing:
    print('OUTPUT_DIR listing:')
    if OUTPUT_DIR.exists():
        for p in sorted(OUTPUT_DIR.rglob('*'))[:200]:
            print(p.relative_to(OUTPUT_DIR), 'dir' if p.is_dir() else p.stat().st_size)
    raise FileNotFoundError('Missing result files: ' + ', '.join(str(p) for p in missing))

with open(OUTPUT_DIR / 'locked_test_config.json', 'r', encoding='utf-8') as f:
    locked_test_config = json.load(f)

runtime_config = locked_test_config.get('runtime_config', {})
print('Runtime config:', runtime_config)
print('Execution mode:', locked_test_config.get('execution_mode'))

expected_runtime = {
    'lookback': 10,
    'sample_mode': 'sliding-window',
    'horizons': [10, 50, 100],
    'seeds': [42, 43, 44],
    'target_scale': 100.0,
    'fixed_len': 100,
}

expected_audit = {
    'execution_mode': 'checkpoint_inference_only',
    'training_performed': False,
    'validation_selection_performed': False,
    'hyperparameter_tuning_performed': False,
    'batch_specific_fine_tuning_performed': False,
    'validation_predictions_written': True,
}

for key, expected in expected_audit.items():
    actual = locked_test_config.get(key)
    if actual != expected:
        raise RuntimeError(f'Unexpected locked_test_config[{key!r}]: expected {expected!r}, got {actual!r}')

for key, expected in expected_runtime.items():
    actual = runtime_config.get(key)
    if actual != expected:
        raise RuntimeError(f'Unexpected runtime_config[{key!r}]: expected {expected!r}, got {actual!r}')

for seed in expected_runtime['seeds']:
    split_path = OUTPUT_DIR / f'split_manifest_seed{seed}.json'
    with open(split_path, 'r', encoding='utf-8') as f:
        split_manifest = json.load(f)
    if split_manifest.get('reused_without_resplitting') is not True:
        raise RuntimeError(f'Original split was not reused: {split_path}')

print('Inference-only audit passed: no training/tuning and original battery splits were reused.')
print('Required model result files exist and runtime config matches the intended sliding-window l10 h10/h50/h100 setup.')


## 7. 기존 모델 vs 개선 모델 성능 비교


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

REPORT_DIR = OUTPUT_DIR / 'report_style_anomaly_analysis'
FIGURE_DIR = REPORT_DIR / 'figures'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

summary_df = pd.read_csv(OUTPUT_DIR / 'locked_test_summary.csv')
horizon_df = pd.read_csv(OUTPUT_DIR / 'test_summary_by_model_horizon.csv')
pred = pd.read_csv(OUTPUT_DIR / 'test_predictions.csv')
val_pred = pd.read_csv(OUTPUT_DIR / 'validation_predictions.csv')

model_order = ['persistence', 'log_degradation', 'cpmlp', 'cpdsconv', 'cpmlp_cpdsconv_fusion']
model_labels = {
    'persistence': 'Persistence',
    'log_degradation': 'Log Degradation',
    'cpmlp': 'CPMLP',
    'cpdsconv': 'CPDSConv',
    'cpmlp_cpdsconv_fusion': 'CPMLP-CPDSConv Fusion',
}
model_colors = {
    'persistence': '#9CA3AF',
    'log_degradation': '#F59E0B',
    'cpmlp': '#10B981',
    'cpdsconv': '#7C3AED',
    'cpmlp_cpdsconv_fusion': '#2563EB',
}

available_order = [m for m in model_order if m in set(summary_df['model'])]
perf = summary_df[summary_df['model'].isin(available_order)].copy()
perf = perf.set_index('model').loc[available_order].reset_index()

display_cols = [c for c in ['model', 'avg_MAE_mean', 'avg_RMSE_mean', 'avg_MAPE_percent_mean', 'average_Skill_MAE_vs_persistence'] if c in perf.columns]
display(perf[display_cols])

if {'cpmlp', 'cpmlp_cpdsconv_fusion'}.issubset(set(summary_df['model'])):
    cpmlp_row = summary_df[summary_df['model'] == 'cpmlp'].iloc[0]
    fusion_row = summary_df[summary_df['model'] == 'cpmlp_cpdsconv_fusion'].iloc[0]
    compare_rows = []
    for col, label in [('avg_MAE_mean', 'MAE'), ('avg_RMSE_mean', 'RMSE'), ('avg_MAPE_percent_mean', 'MAPE')]:
        if col in summary_df.columns:
            compare_rows.append({
                'metric': label,
                'cpmlp': cpmlp_row[col],
                'cpmlp_cpdsconv_fusion': fusion_row[col],
                'delta_percent_vs_cpmlp': (cpmlp_row[col] - fusion_row[col]) / cpmlp_row[col] * 100,
            })
    comparison_df = pd.DataFrame(compare_rows)
    display(comparison_df)
    comparison_df.to_csv(REPORT_DIR / 'fusion_vs_cpmlp_metrics.csv', index=False, encoding='utf-8-sig')

metric_cols = [('avg_MAE_mean', 'MAE'), ('avg_RMSE_mean', 'RMSE'), ('avg_MAPE_percent_mean', 'MAPE (%)')]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (col, label) in zip(axes, metric_cols):
    if col not in perf.columns:
        ax.set_visible(False)
        continue
    bars = ax.bar(
        perf['model'].map(model_labels),
        perf[col],
        color=[model_colors[m] for m in perf['model']],
    )
    ax.set_title(label)
    ax.set_ylabel(label)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=18)
    for bar in bars:
        h = bar.get_height()
        txt = f'{h:.5f}' if label != 'MAPE (%)' else f'{h:.3f}'
        ax.text(bar.get_x() + bar.get_width()/2, h, txt, ha='center', va='bottom', fontsize=9)
fig.suptitle('Model Performance Comparison: Baselines vs Fusion Model', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / '01_model_performance_comparison.png', dpi=220)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
hmetric_cols = [('MAE_mean', 'MAE'), ('RMSE_mean', 'RMSE'), ('MAPE_percent_mean', 'MAPE (%)')]
for ax, (col, label) in zip(axes, hmetric_cols):
    if col not in horizon_df.columns:
        ax.set_visible(False)
        continue
    for model in available_order:
        sub = horizon_df[horizon_df['model'] == model].sort_values('horizon')
        if sub.empty:
            continue
        ax.plot(sub['horizon'], sub[col], marker='o', linewidth=2, label=model_labels[model], color=model_colors[model])
    ax.set_title(label)
    ax.set_xlabel('Prediction Horizon')
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('Performance by Horizon', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / '02_performance_by_horizon.png', dpi=220)
plt.show()


## 8. Residual 기반 셀 단위 empirical conformal 이상탐지


In [ ]:
# 1) Build residual components from validation and locked-test predictions.
validation_df = prepare_residual_features(
    val_pred,
    'validation',
    target_model=ANOMALY_CONFIG.target_model,
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
)
df = prepare_residual_features(
    pred,
    'test',
    target_model=ANOMALY_CONFIG.target_model,
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
)
assert_seed_split_isolation(validation_df, df)

# 2) Use about one third of validation cells for score development and keep
#    about two thirds untouched as the empirical conformal reference set.
validation_df, validation_role_table = split_validation_roles(
    validation_df,
    selection_fraction=ANOMALY_CONFIG.validation_selection_fraction,
    random_state=ANOMALY_CONFIG.random_state,
)
alpha_selection_raw = validation_df[
    validation_df['validation_role'] == 'alpha_selection'
].copy()

# 3) Component scaling and alpha selection use alpha-development cells only.
component_calibration = fit_component_calibration(alpha_selection_raw)
validation_df = apply_component_calibration(validation_df, component_calibration)
df = apply_component_calibration(df, component_calibration)
alpha_selection_df = validation_df[
    validation_df['validation_role'] == 'alpha_selection'
].copy()
alpha_by_seed, alpha_search_table = select_alpha_by_seed(
    alpha_selection_df,
    alpha_grid=ANOMALY_CONFIG.alpha_grid,
    horizons=ANOMALY_CONFIG.alpha_selection_horizons,
    severity_aggregation=ANOMALY_CONFIG.alpha_cell_aggregation,
    min_common_windows=ANOMALY_CONFIG.min_common_windows,
    min_paired_cells=ANOMALY_CONFIG.min_paired_cells,
    bootstrap_repeats=ANOMALY_CONFIG.bootstrap_repeats,
    random_state=ANOMALY_CONFIG.random_state,
)
validation_df = add_scores_by_seed(validation_df, alpha_by_seed)
df = add_scores_by_seed(df, alpha_by_seed)

# 4) Convert overlapping windows into one continuous mean severity per
#    cell and horizon, using only input_end_cycle values common to all horizons.
alpha_horizon_summary = summarize_common_horizon_scores(
    validation_df[validation_df['validation_role'] == 'alpha_selection'],
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
    min_common_windows=ANOMALY_CONFIG.min_common_windows,
)
calibration_horizon_summary = summarize_common_horizon_scores(
    validation_df[validation_df['validation_role'] == 'conformal_calibration'],
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
    min_common_windows=ANOMALY_CONFIG.min_common_windows,
)
cell_horizon_summary = summarize_common_horizon_scores(
    df,
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
    min_common_windows=ANOMALY_CONFIG.min_common_windows,
)

# 5) Put H10/H50/H100 cell means on comparable alpha-development scales.
horizon_severity_calibration = fit_horizon_severity_calibration(
    alpha_horizon_summary
)
calibration_horizon_summary = apply_horizon_severity_calibration(
    calibration_horizon_summary, horizon_severity_calibration
)
cell_horizon_summary = apply_horizon_severity_calibration(
    cell_horizon_summary, horizon_severity_calibration
)

# 6) The median of the three relative horizon means is one continuous
#    cell nonconformity score. No window flag, ratio, or binary horizon vote.
calibration_cell_summary = aggregate_cell_nonconformity(
    calibration_horizon_summary,
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
)
unscored_test_cells = aggregate_cell_nonconformity(
    cell_horizon_summary,
    expected_horizons=ANOMALY_CONFIG.expected_horizons,
)
cell_summary, conformal_audit = apply_empirical_conformal_pvalues(
    unscored_test_cells,
    calibration_cell_summary,
    candidate_p=ANOMALY_CONFIG.conformal_candidate_p,
    strong_p=ANOMALY_CONFIG.conformal_strong_p,
)
physical_cell_summary = aggregate_physical_cell_evidence(cell_summary)
coverage_sensitivity_audit = pd.concat(
    [
        audit_cell_score_coverage(
            calibration_cell_summary, data_role='conformal_calibration'
        ),
        audit_cell_score_coverage(cell_summary, data_role='locked_test'),
    ],
    ignore_index=True,
)
batch_candidate_audit = (
    physical_cell_summary.groupby('batch_id', as_index=False)
    .agg(
        evaluated_physical_cells=('battery_id', 'nunique'),
        empirical_tail_candidates=('is_physical_candidate', 'sum'),
        strong_tail_candidates=('is_physical_strong_candidate', 'sum'),
        repeated_seed_candidates=('is_repeated_seed_candidate', 'sum'),
        single_seed_candidates=('is_single_seed_candidate', 'sum'),
        median_worst_seed_p=('worst_seed_empirical_p_value', 'median'),
    )
)
batch_candidate_audit['candidate_rate'] = (
    batch_candidate_audit['empirical_tail_candidates']
    / batch_candidate_audit['evaluated_physical_cells']
)

# 7) Save score provenance, calibration references, and both seed-level and
#    unique physical-cell result tables.
validation_role_table.to_csv(
    REPORT_DIR / 'validation_battery_roles.csv', index=False, encoding='utf-8-sig'
)
alpha_by_seed.to_csv(
    REPORT_DIR / 'selected_anomaly_alpha_by_seed.csv', index=False, encoding='utf-8-sig'
)
alpha_search_table.to_csv(
    REPORT_DIR / 'anomaly_alpha_search_validation.csv', index=False, encoding='utf-8-sig'
)
component_calibration.to_csv(
    REPORT_DIR / 'alpha_development_component_calibration.csv', index=False, encoding='utf-8-sig'
)
horizon_severity_calibration.to_csv(
    REPORT_DIR / 'alpha_development_horizon_calibration.csv', index=False, encoding='utf-8-sig'
)
calibration_cell_summary.to_csv(
    REPORT_DIR / 'validation_cell_conformal_calibration.csv', index=False, encoding='utf-8-sig'
)
conformal_audit.to_csv(
    REPORT_DIR / 'conformal_calibration_audit.csv', index=False, encoding='utf-8-sig'
)
coverage_sensitivity_audit.to_csv(
    REPORT_DIR / 'cell_score_coverage_sensitivity_audit.csv', index=False, encoding='utf-8-sig'
)
df.to_csv(REPORT_DIR / 'cycle_level_degradation_anomaly_scores.csv', index=False, encoding='utf-8-sig')
cell_horizon_summary.to_csv(REPORT_DIR / 'cell_horizon_mean_severity.csv', index=False, encoding='utf-8-sig')
cell_summary.to_csv(REPORT_DIR / 'cell_level_conformal_summary_by_seed.csv', index=False, encoding='utf-8-sig')
physical_cell_summary.to_csv(REPORT_DIR / 'physical_cell_conformal_summary.csv', index=False, encoding='utf-8-sig')
batch_candidate_audit.to_csv(REPORT_DIR / 'physical_cell_conformal_summary_by_batch.csv', index=False, encoding='utf-8-sig')

alpha_figure = plot_alpha_search(
    alpha_search_table,
    alpha_by_seed,
    save_path=FIGURE_DIR / '03_validation_only_alpha_selection.png',
)
plt.show()

print('Stability-selected alpha by seed:')
display(alpha_by_seed)
print('Empirical conformal calibration audit:')
display(conformal_audit)
print('Coverage sensitivity audit (diagnostic only):')
display(coverage_sensitivity_audit)
if coverage_sensitivity_audit['has_large_coverage_association'].any():
    print('WARNING: score rarity is strongly associated with observed life coverage for at least one seed/role; inspect raw SOH curves before interpreting it as abnormal degradation.')
print('Seed-level empirical tail candidates:', int(cell_summary['is_conformal_candidate'].sum()), '/', len(cell_summary))
print('Seed-level strong candidates:', int(cell_summary['is_strong_candidate'].sum()))
print('Unique physical cells evaluated:', len(physical_cell_summary))
print('Physical-cell candidates across all available test seeds:', int(physical_cell_summary['is_physical_candidate'].sum()))
print('Repeated-seed candidates:', int(physical_cell_summary['is_repeated_seed_candidate'].sum()))
print('Single-test-seed candidates:', int(physical_cell_summary['is_single_seed_candidate'].sum()))
print('Batch-level candidate audit (pooled p-values are marginal, not batch-conditional):')
display(batch_candidate_audit)
print('Saved:', REPORT_DIR)
display(physical_cell_summary.head(30))


## 9. 셀 단위 empirical conformal 후보 결과 그래프


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# Load analysis results
# =========================
REPORT_DIR = OUTPUT_DIR / 'report_style_anomaly_analysis'
FIGURE_DIR = REPORT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

cycle_path = REPORT_DIR / 'cycle_level_degradation_anomaly_scores.csv'
cell_path = REPORT_DIR / 'cell_level_conformal_summary_by_seed.csv'
physical_path = REPORT_DIR / 'physical_cell_conformal_summary.csv'
alpha_path = REPORT_DIR / 'selected_anomaly_alpha_by_seed.csv'

df = pd.read_csv(cycle_path)
cell_summary = pd.read_csv(cell_path)
physical_cell_summary = pd.read_csv(physical_path)
alpha_by_seed = pd.read_csv(alpha_path)

# =========================
# Candidate selection
# =========================
cand = physical_cell_summary[
    physical_cell_summary['is_physical_strong_candidate'] == True
].copy()
candidate_view = 'strong empirical-tail physical-cell candidates'
if cand.empty:
    cand = physical_cell_summary[
        physical_cell_summary['is_physical_candidate'] == True
    ].copy()
    candidate_view = 'empirical-tail physical-cell candidates'
if cand.empty:
    cand = physical_cell_summary.copy()
    candidate_view = 'exploratory rarity ranking; no p-cutoff candidate'

required_conformal_cols = {
    'median_h10_relative_severity', 'median_h50_relative_severity',
    'median_h100_relative_severity', 'median_cell_nonconformity_score',
    'worst_seed_empirical_p_value', 'is_physical_candidate',
    'is_physical_strong_candidate', 'physical_cell_status',
    'is_repeated_seed_candidate', 'is_repeated_seed_strong_candidate',
    'is_single_seed_candidate', 'is_single_seed_strong_candidate',
}
missing_conformal_cols = required_conformal_cols - set(cand.columns)
if missing_conformal_cols:
    raise RuntimeError(f'Missing conformal columns: {sorted(missing_conformal_cols)}')

cand = cand.sort_values(
    ['is_repeated_seed_strong_candidate', 'is_repeated_seed_candidate',
     'is_single_seed_strong_candidate', 'is_single_seed_candidate',
     'worst_seed_empirical_p_value', 'median_cell_nonconformity_score'],
    ascending=[False, False, False, False, True, False],
).head(12).reset_index(drop=True)
cand['label'] = (
    cand['battery_id'].astype(str)
    + '\nseeds=' + cand['test_seeds'].astype(str)
    + '\n' + cand['physical_cell_status'].astype(str)
)
cand['status_color'] = cand['physical_cell_status'].map({
    'repeated_seed_strong_empirical_tail_candidate': '#991B1B',
    'repeated_seed_empirical_tail_candidate': '#DC2626',
    'single_seed_strong_empirical_tail_candidate': '#F97316',
    'single_seed_empirical_tail_candidate': '#F59E0B',
}).fillna('#64748B')

display(cand)

# =========================
# Plot
# =========================
fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))

x = np.arange(len(cand))
width = 0.25
for offset, horizon, color in [
    (-width, 10, '#60A5FA'), (0.0, 50, '#8B5CF6'), (width, 100, '#EF4444')
]:
    axes[0].bar(
        x + offset, cand[f'median_h{horizon}_relative_severity'],
        width=width, label=f'H{horizon}', color=color, alpha=0.85
    )
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xticks(x, cand['label'], rotation=50)
axes[0].set_title('Relative Mean Severity by Horizon')
axes[0].set_ylabel('Alpha-development relative severity')
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend()

axes[1].bar(
    cand['label'], cand['median_cell_nonconformity_score'],
    color=cand['status_color']
)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Cell Nonconformity Score')
axes[1].set_ylabel('Median of H10/H50/H100 relative means')
axes[1].tick_params(axis='x', rotation=50)
axes[1].grid(axis='y', alpha=0.3)

plot_p = cand['worst_seed_empirical_p_value'].clip(lower=1e-12)
plot_evidence = -np.log10(plot_p)
axes[2].bar(cand['label'], plot_evidence, color=cand['status_color'])
axes[2].axhline(
    -np.log10(ANOMALY_CONFIG.conformal_candidate_p),
    color='#F97316', linestyle='--', linewidth=1.5,
    label=f'candidate p≤{ANOMALY_CONFIG.conformal_candidate_p:.2f}'
)
axes[2].axhline(
    -np.log10(ANOMALY_CONFIG.conformal_strong_p),
    color='#DC2626', linestyle=':', linewidth=1.8,
    label=f'strong p≤{ANOMALY_CONFIG.conformal_strong_p:.2f}'
)
axes[2].set_title('Empirical Tail Evidence')
axes[2].set_ylabel('-log10(worst-seed p-value)')
axes[2].tick_params(axis='x', rotation=50)
axes[2].grid(axis='y', alpha=0.3)
axes[2].legend()
for i, p_value in enumerate(plot_p):
    axes[2].text(i, plot_evidence.iloc[i] + 0.03, f'p={p_value:.3f}', ha='center', fontsize=8)

alpha_label = ', '.join(
    f'seed{int(row.seed)} alpha={row.alpha:.2f}'
    for row in alpha_by_seed.itertuples(index=False)
)
fig.suptitle(
    f'{candidate_view} with stability-selected alpha ({alpha_label})',
    fontsize=16,
)
plt.tight_layout()

save_path = FIGURE_DIR / 'physical_cell_empirical_conformal_candidates.png'
plt.savefig(save_path, dpi=220)
plt.show()

print('saved:', save_path)

In [ ]:
import pickle
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path
import soh_gru_dsconv_pipeline as soh_pipe

REPORT_DIR = OUTPUT_DIR / 'report_style_anomaly_analysis'
FIGURE_DIR = REPORT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

physical_cell_summary = pd.read_csv(REPORT_DIR / 'physical_cell_conformal_summary.csv')
if 'df' not in globals():
    df = pd.read_csv(REPORT_DIR / 'cycle_level_degradation_anomaly_scores.csv')

def extract_batch_id(cell_id):
    m = re.search(r'(b\d+)', str(cell_id))
    return m.group(1) if m else 'unknown'

def extract_cell_id_from_path(path):
    return Path(path).stem

def load_matr_soh_curve(pkl_path, debug=False):
    with open(pkl_path, 'rb') as f:
        obj = pickle.load(f)

    if not isinstance(obj, dict):
        raise ValueError(f'Unsupported pkl type: {type(obj)}')

    if 'cycle_data' not in obj:
        raise ValueError(f'No cycle_data key. keys={list(obj.keys())[:20]}')

    cycles = list(obj['cycle_data'])
    cycles = sorted(cycles, key=lambda cyc: int(cyc.get('cycle_number', len(cycles))))
    reference_capacity = float(soh_pipe.infer_reference_capacity(obj, cycles))
    battery_id = Path(pkl_path).stem
    cell_id = str(obj.get('cell_id', battery_id))
    batch_id = extract_batch_id(battery_id)

    rows = []
    skipped = []
    for idx, cyc in enumerate(cycles):
        cycle_num = int(cyc.get('cycle_number', idx + 1))
        try:
            # Use the exact target definition used by the SOH model:
            # max(discharge_capacity_in_Ah) / reference_capacity.
            soh = float(soh_pipe.extract_soh_label(cyc, reference_capacity))
        except Exception as exc:
            skipped.append((cycle_num, str(exc)))
            continue

        rows.append({
            'cycle': cycle_num,
            'soh': soh,
            'capacity': soh * reference_capacity,
            'reference_capacity': reference_capacity,
            'battery_id': battery_id,
            'cell_id': cell_id,
            'batch_id': batch_id,
        })

    if not rows:
        raise ValueError('No SOH labels extracted with the model target definition')

    # The model stores SOH by cycle number, so a repeated cycle keeps the last value.
    curve = (
        pd.DataFrame(rows)
        .drop_duplicates('cycle', keep='last')
        .sort_values('cycle')
        .reset_index(drop=True)
    )

    if debug:
        print('file:', pkl_path)
        print('battery_id:', battery_id, 'cell_id:', cell_id)
        print('reference_capacity:', reference_capacity)
        print('model-target SOH points:', len(curve), 'skipped:', len(skipped))
        if skipped[:5]:
            print('skipped examples:', skipped[:5])

    return curve

# =========================
# 1. 후보 cell 선택
# =========================
physical_cell_summary['batch_id'] = physical_cell_summary['battery_id'].apply(extract_batch_id)

candidate_cells = physical_cell_summary[
    physical_cell_summary['is_physical_strong_candidate'] == True
].copy()
raw_curve_view = 'strong empirical-tail candidate'
if candidate_cells.empty:
    candidate_cells = physical_cell_summary[
        physical_cell_summary['is_physical_candidate'] == True
    ].copy()
    raw_curve_view = 'empirical-tail candidate'
if candidate_cells.empty:
    candidate_cells = physical_cell_summary.copy()
    raw_curve_view = 'exploratory rarity ranking; no p-cutoff candidate'
candidate_cells = candidate_cells.sort_values(
    ['is_repeated_seed_strong_candidate', 'is_repeated_seed_candidate',
     'is_single_seed_strong_candidate', 'is_single_seed_candidate',
     'worst_seed_empirical_p_value', 'median_cell_nonconformity_score'],
    ascending=[False, False, False, False, True, False],
).copy()

target_batch = candidate_cells.iloc[0]['batch_id']
candidate_cells = candidate_cells[
    candidate_cells['batch_id'] == target_batch
].head(2).copy()
candidate_ids = sorted(candidate_cells['battery_id'].astype(str).unique())
evaluated_ids = set(physical_cell_summary['battery_id'].astype(str))
all_candidate_ids = set(
    physical_cell_summary.loc[
        physical_cell_summary['is_physical_candidate'] == True, 'battery_id'
    ].astype(str)
)
candidate_status_by_id = (
    candidate_cells.drop_duplicates('battery_id')
    .set_index('battery_id')['physical_cell_status']
    .astype(str)
    .to_dict()
)

print('target_batch:', target_batch)
print('raw_curve_view:', raw_curve_view)
print('candidate_battery_ids:', candidate_ids)
display(candidate_cells)

# =========================
# 2. 같은 batch raw pkl curve 로드
# =========================
pkl_paths = sorted(MATR_DIR.rglob('*.pkl'))

sample = next(
    (p for p in pkl_paths if extract_batch_id(extract_cell_id_from_path(p)) == target_batch),
    None
)

if sample is not None:
    _ = load_matr_soh_curve(sample, debug=True)

records = []
failed = []

for p in pkl_paths:
    battery_id = extract_cell_id_from_path(p)
    batch_id = extract_batch_id(battery_id)

    if batch_id != target_batch:
        continue

    try:
        curve = load_matr_soh_curve(p)
        records.append(curve)
    except Exception as e:
        failed.append((str(p), str(e)))

print('loaded curves:', len(records))
print('failed curves:', len(failed))

if failed[:5]:
    print('failed examples:')
    for item in failed[:5]:
        print(item)

if not records:
    raise RuntimeError('No raw SOH curves loaded.')

raw_curves = pd.concat(records, ignore_index=True)

print('same batch batteries:', raw_curves['battery_id'].nunique())
print('cycle range:', raw_curves['cycle'].min(), raw_curves['cycle'].max())

# Verify that plotted SOH and model actual_soh use the same target definition.
parity_required = {'battery_id', 'target_cycle', 'actual_soh'}
if parity_required.issubset(df.columns):
    parity = (
        df[['battery_id', 'target_cycle', 'actual_soh']]
        .drop_duplicates()
        .merge(
            raw_curves[['battery_id', 'cycle', 'soh']].drop_duplicates(),
            left_on=['battery_id', 'target_cycle'],
            right_on=['battery_id', 'cycle'],
            how='inner',
            validate='many_to_one',
        )
    )
    if parity.empty:
        raise RuntimeError('Could not match plotted SOH to model actual_soh for parity audit')
    parity['abs_diff'] = (parity['actual_soh'] - parity['soh']).abs()
    max_abs_diff = float(parity['abs_diff'].max())
    print('SOH target parity rows:', len(parity), 'max_abs_diff:', max_abs_diff)
    if max_abs_diff > 1e-6:
        raise RuntimeError(f'Raw SOH/model target mismatch: max_abs_diff={max_abs_diff}')

# =========================
# 3. Plot: full observed cycle/SOH range
# =========================
plt.figure(figsize=(11, 6))

for battery_id, one in raw_curves.groupby('battery_id'):
    one = one.sort_values('cycle')

    if battery_id in candidate_ids:
        continue

    if battery_id in all_candidate_ids:
        peer_style = dict(color='#FDBA74', linewidth=1.3, alpha=0.65, linestyle='--')
    elif battery_id in evaluated_ids:
        peer_style = dict(color='#9CA3AF', linewidth=1.0, alpha=0.55, linestyle='-')
    else:
        peer_style = dict(color='#E5E7EB', linewidth=0.7, alpha=0.30, linestyle=':')
    plt.plot(one['cycle'], one['soh'], **peer_style)

candidate_colors = ['#2563EB', '#F97316', '#DC2626', '#16A34A']

for idx, battery_id in enumerate(candidate_ids):
    one = raw_curves[raw_curves['battery_id'] == battery_id].sort_values('cycle')

    if one.empty:
        print('candidate raw curve not found:', battery_id)
        continue

    plt.plot(
        one['cycle'],
        one['soh'],
        color=candidate_colors[idx % len(candidate_colors)],
        linewidth=2.8,
        label=f"{battery_id} [{candidate_status_by_id.get(battery_id, raw_curve_view)}]"
    )

plt.xlabel('Cycle')
plt.ylabel('SOH (model target definition)')
plt.title(f'SOH Curves: {target_batch} selected cells vs non-selected peers ({raw_curve_view})')
plt.axhline(1.0, color='#64748B', linestyle=':', linewidth=1.0, label='SOH=1.0')
plt.axhline(0.8, color='#94A3B8', linestyle='--', linewidth=1.0, label='SOH=0.8')
plt.margins(x=0.01, y=0.05)
plt.grid(alpha=0.3)
handles, labels = plt.gca().get_legend_handles_labels()
peer_handles = [
    Line2D([0], [0], color='#FDBA74', linestyle='--', label='other selected candidate'),
    Line2D([0], [0], color='#9CA3AF', linestyle='-', label='evaluated non-selected peer'),
    Line2D([0], [0], color='#E5E7EB', linestyle=':', label='not evaluated by locked test'),
]
plt.legend(handles=handles + peer_handles)
plt.tight_layout()

save_path = FIGURE_DIR / f'batch_{target_batch}_top_candidates_vs_peers_full_observed_range.png'
plt.savefig(save_path, dpi=220)
plt.show()

print('saved:', save_path)

In [ ]:
# =========================
# b3 batch conformal candidates vs non-selected peers
# =========================

target_batch = 'b3'

if 'load_matr_soh_curve' not in globals():
    raise RuntimeError('Run the preceding model-parity raw SOH helper cell first.')
physical_cell_summary = pd.read_csv(
    REPORT_DIR / 'physical_cell_conformal_summary.csv'
)
physical_cell_summary['batch_id'] = (
    physical_cell_summary['battery_id'].apply(extract_batch_id)
)

# Select strong empirical-tail candidates first, then p<=candidate cutoff.
b3_candidates = physical_cell_summary[
    (physical_cell_summary['batch_id'] == target_batch) &
    (physical_cell_summary['is_physical_strong_candidate'] == True)
].copy()
b3_curve_view = 'strong empirical-tail candidate'
if b3_candidates.empty:
    b3_candidates = physical_cell_summary[
        (physical_cell_summary['batch_id'] == target_batch)
        & (physical_cell_summary['is_physical_candidate'] == True)
    ].copy()
    b3_curve_view = 'empirical-tail candidate'
if b3_candidates.empty:
    b3_candidates = physical_cell_summary[
        physical_cell_summary['batch_id'] == target_batch
    ].copy()
    b3_curve_view = 'exploratory rarity ranking; no p-cutoff candidate'
b3_candidates = (
    b3_candidates
    .sort_values(
        ['is_repeated_seed_strong_candidate', 'is_repeated_seed_candidate',
         'is_single_seed_strong_candidate', 'is_single_seed_candidate',
         'worst_seed_empirical_p_value', 'median_cell_nonconformity_score'],
        ascending=[False, False, False, False, True, False],
    )
    .head(2)
    .copy()
)

b3_candidate_ids = sorted(b3_candidates['battery_id'].astype(str).unique())
b3_evaluated_ids = set(physical_cell_summary['battery_id'].astype(str))
b3_all_candidate_ids = set(
    physical_cell_summary.loc[
        physical_cell_summary['is_physical_candidate'] == True, 'battery_id'
    ].astype(str)
)
b3_status_by_id = (
    b3_candidates.drop_duplicates('battery_id')
    .set_index('battery_id')['physical_cell_status']
    .astype(str)
    .to_dict()
)

print('target_batch:', target_batch)
print('b3_curve_view:', b3_curve_view)
print('b3_candidate_battery_ids:', b3_candidate_ids)
display(b3_candidates)

# b3 raw curves 로드
pkl_paths = sorted(MATR_DIR.rglob('*.pkl'))

records = []
failed = []

for p in pkl_paths:
    battery_id = extract_cell_id_from_path(p)
    batch_id = extract_batch_id(battery_id)

    if batch_id != target_batch:
        continue

    try:
        curve = load_matr_soh_curve(p)
        records.append(curve)
    except Exception as e:
        failed.append((str(p), str(e)))

print('loaded b3 curves:', len(records))
print('failed b3 curves:', len(failed))

if failed[:5]:
    print('failed examples:')
    for item in failed[:5]:
        print(item)

if not records:
    raise RuntimeError('No b3 raw SOH curves loaded.')

b3_raw_curves = pd.concat(records, ignore_index=True)

print('b3 batteries:', b3_raw_curves['battery_id'].nunique())
print('b3 cycle range:', b3_raw_curves['cycle'].min(), b3_raw_curves['cycle'].max())

# plot
plt.figure(figsize=(11, 6))

# Non-selected peers: gray (not proven normal).
for battery_id, one in b3_raw_curves.groupby('battery_id'):
    one = one.sort_values('cycle')

    if battery_id in b3_candidate_ids:
        continue

    if battery_id in b3_all_candidate_ids:
        peer_style = dict(color='#FDBA74', linewidth=1.3, alpha=0.65, linestyle='--')
    elif battery_id in b3_evaluated_ids:
        peer_style = dict(color='#9CA3AF', linewidth=1.0, alpha=0.55, linestyle='-')
    else:
        peer_style = dict(color='#E5E7EB', linewidth=0.7, alpha=0.30, linestyle=':')
    plt.plot(one['cycle'], one['soh'], **peer_style)

# candidates: colored
candidate_colors = ['#2563EB', '#F97316', '#DC2626', '#16A34A']

for idx, battery_id in enumerate(b3_candidate_ids):
    one = b3_raw_curves[b3_raw_curves['battery_id'] == battery_id].sort_values('cycle')

    if one.empty:
        print('candidate raw curve not found:', battery_id)
        continue

    plt.plot(
        one['cycle'],
        one['soh'],
        color=candidate_colors[idx % len(candidate_colors)],
        linewidth=2.8,
        label=f"{battery_id} [{b3_status_by_id.get(battery_id, b3_curve_view)}]"
    )

plt.xlabel('Cycle')
plt.ylabel('SOH (model target definition)')
plt.title(f'SOH Curves: {target_batch} selected cells vs non-selected peers ({b3_curve_view})')
plt.axhline(1.0, color='#64748B', linestyle=':', linewidth=1.0, label='SOH=1.0')
plt.axhline(0.8, color='#94A3B8', linestyle='--', linewidth=1.0, label='SOH=0.8')
plt.margins(x=0.01, y=0.05)
plt.grid(alpha=0.3)
handles, labels = plt.gca().get_legend_handles_labels()
peer_handles = [
    Line2D([0], [0], color='#FDBA74', linestyle='--', label='other selected candidate'),
    Line2D([0], [0], color='#9CA3AF', linestyle='-', label='evaluated non-selected peer'),
    Line2D([0], [0], color='#E5E7EB', linestyle=':', label='not evaluated by locked test'),
]
plt.legend(handles=handles + peer_handles)
plt.tight_layout()

save_path = FIGURE_DIR / f'batch_{target_batch}_dedicated_candidates_vs_peers_full_observed_range.png'
plt.savefig(save_path, dpi=220)
plt.show()

print('saved:', save_path)